# Bilingual Voice Benchmark Demo

This notebook demonstrates the VoxTool MVP end to end with deterministic mock
adapters: Russian and English text examples, synthesized audio, pipelines A-D,
parsed JSON envelopes, optional `units.convert` execution, metrics, and the
final markdown report.

**Prerequisites**: install dependencies with `make install` (or
`uv sync --all-groups`). No GPU, cloud service, paid API, or real model
download is required. The notebook uses the committed fixture dataset by
default and falls back to `data/generated/v1/examples.jsonl` when present.
All generated artifacts are written under `runs/notebook/` and stay outside Git.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Repository root with pyproject.toml not found.")
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from apps.notebook import helpers  # noqa: E402

GENERATED_DATASET = ROOT / "data" / "generated" / "v1" / "examples.jsonl"
FIXTURE_DATASET = ROOT / "data" / "fixtures" / "examples.small.jsonl"
DATASET_PATH = GENERATED_DATASET if GENERATED_DATASET.exists() else FIXTURE_DATASET
OUTPUT_DIR = ROOT / "runs" / "notebook"
AUDIO_DIR = OUTPUT_DIR / "audio"

print(f"dataset: {DATASET_PATH.relative_to(ROOT)}")
print(f"artifacts: {OUTPUT_DIR.relative_to(ROOT)}")

## 1. Russian And English Text Examples

The dataset contains bilingual `units.convert` examples plus no-tool examples.
We select one tool and one no-tool example per language for display.

In [ ]:
examples = helpers.load_examples(DATASET_PATH)
display_examples = helpers.select_bilingual_examples(examples)
print(f"loaded {len(examples)} examples")
helpers.examples_table(display_examples)

## 2. Synthesize Audio Counterparts

Every text example gets a deterministic local audio fixture and a metadata
record linking `audio_id` to `example_id` with the synthesis settings.

In [ ]:
audio_metadata_path, audio_records = helpers.synthesize_audio_metadata(
    examples, AUDIO_DIR
)
metadata_rel = audio_metadata_path.relative_to(ROOT)
print(f"wrote {len(audio_records)} audio records to {metadata_rel}")
audio_records[0].model_dump(mode="json")

## 3. Run Pipelines A-D With Mock Adapters

- **A**: text in, JSON envelope out
- **B**: audio in, ASR transcript out (WER against the reference transcript)
- **C**: audio in, direct audio tool-use JSON envelope out
- **D**: audio in, ASR transcript then text-LLM JSON envelope out

Tool manifests come from `ToolRegistry`; validated calls execute only through
`ToolExecutor`. Raw outputs, parse status, repair attempts, validation errors,
and structured failures are recorded for every example.

In [ ]:
records_by_pipeline = helpers.run_mock_pipelines(
    dataset_path=DATASET_PATH,
    audio_metadata_path=audio_metadata_path,
    output_dir=OUTPUT_DIR,
    run_id="notebook-demo",
)
for pipeline, records in records_by_pipeline.items():
    print(f"Pipeline {pipeline}: {len(records)} records")
helpers.records_table(records_by_pipeline["A"])

## 4. Parsed JSON Envelope And Optional Tool Execution

Each model output is the canonical JSON envelope. Validated `units.convert`
calls are optionally executed and the deterministic result feeds the final
answer.

In [ ]:
for example in display_examples:
    record = next(
        r for r in records_by_pipeline["A"] if r.example_id == example.example_id
    )
    print(f"--- {example.example_id} ({example.language}) ---")
    print(f"text: {example.text}")
    print(f"parsed envelope: {record.parsed_output}")
    if record.tool_execution_result is not None:
        print(f"tool execution: {record.tool_execution_result.model_dump(mode='json')}")
    print(f"final answer: {record.final_answer}")
    print()

## 5. Audio Transcript Output And Audio Tool Calling

Pipeline B records transcripts with per-example WER. Pipelines C and D show
audio tool calling directly and via the ASR-then-text route.

In [ ]:
helpers.records_table(records_by_pipeline["B"])

In [ ]:
import pandas as pd

pd.concat(
    [
        helpers.records_table(records_by_pipeline["C"]),
        helpers.records_table(records_by_pipeline["D"]),
    ],
    ignore_index=True,
)

## 6. Metrics Summary

Aggregated per pipeline with overall, per-split, and per-language rows,
including the modality gap of audio pipelines against the Pipeline A baseline.

In [ ]:
summary = helpers.metrics_summary(
    examples=examples, records_by_pipeline=records_by_pipeline
)
summary

## 7. Build The Markdown Report

The report includes the dataset summary, per-pipeline metrics, language
splits, tool/no-tool confusion matrix, WER, modality gap, best-pipeline
rationale, and categorized failures.

In [ ]:
from packages.metrics.aggregation import write_summary
from packages.report_builder.report import build_report, write_report

report = build_report(
    dataset=examples, records_by_pipeline=records_by_pipeline, plot_paths=[]
)
report_path = OUTPUT_DIR / "report.md"
summary_path = OUTPUT_DIR / "metrics-summary.csv"
write_report(report_path, report)
write_summary(summary, summary_path)
print(f"report: {report_path.relative_to(ROOT)}")
print(f"metrics summary: {summary_path.relative_to(ROOT)}")
print()
print(report_path.read_text(encoding="utf-8")[:600])

## Artifacts

- Pipeline runs: `runs/notebook/pipeline-{a,b,c,d}.jsonl`
- Audio fixtures and metadata: `runs/notebook/audio/`
- Metrics summary: `runs/notebook/metrics-summary.csv`
- Report: `runs/notebook/report.md`

All of these stay outside Git per the artifact policy. For the full 240-example
flow, follow `specs/002-voice-benchmark-demo/quickstart.md`.